# 函数

> **开始前请注意：在线输入限制**
> 当前在线环境中的 `scanf`、`getchar`、`fgets(..., stdin)` 无法交互读取键盘输入。在线实验请修改变量的初值，或使用 `sscanf` 从字符串读取；键盘输入练习请在本地 GCC / Clang 中运行完整 C 程序。

**本章目标**：区分声明、定义和调用；传入数组及长度；设计明确的错误结果；验证递归边界。

**学习方法**：先预测输出，再运行验证；每次只修改一个条件，最后用自己的话解释变化。修改函数或类型定义后，请重启内核并从头运行。

在 C 语言中，**函数** 是完成特定任务的一段代码，其他代码可以通过函数名多次调用函数。  

它能提高代码的可重用性和可读性。

C 程序从 `main` 函数开始执行。

## 1. 函数定义

```c
返回值类型 函数名(参数列表) {
    // 函数体
    return 返回值;
}
```

## 2. 函数调用

In [ ]:
// 示例：求两个数的和
#include <stdio.h>

// 定义add函数
int add(int a, int b) {
    return a + b;
}

{
    //调用add函数
    int result = add(3, 5);
    printf("结果: %d\n", result);
}

## 3. 函数参数

在 C 中，函数参数都是“值传递”，函数获得的是实参的副本（copy），该副本被称为形参，对形参的修改不会影响实参（原变量）。

In [ ]:
void change(int x) { //此处x称为函数的形参
    x = 10;
    printf("x = %d\n", x);  // x = 10
}

{
    int a = 5;
    change(a); //此处，a称为实参
    printf("a = %d\n", a);  // a = 5 a没有改变
}

使用指针修改实参

In [ ]:
void change2(int *x) {
    *x = 10;
}

{
    int a = 5;
    change2(&a);   //scanf函数传参时与此相同，需要传入变量的地址
    printf("%d\n", a);  // 修改为 10
}

## 4. 函数返回值

函数可以返回一个值，也可以返回 void 表示无返回值。

In [ ]:
int square(int x) {
    return x * x;
}

void hello(int x) {
    printf("Hello, C!\n");
    printf("x = %d\n",x);
}

{
    int y = square(10);
    hello(y);
}

## 5. 变量作用域与存储期

- 块内局部变量的名字只在对应作用域中可见，普通自动局部对象在离开块后不再存在。
- 文件作用域名字通常从声明处开始可见；跨文件访问还涉及声明与链接，并非“所有函数都能随意访问”。
- 作用域回答“名字在哪里可见”，存储期回答“对象何时存在”。初学时优先通过参数和返回值传递数据。

In [ ]:
int global = 100; // 演示文件作用域；函数只读取，不修改它
void show_scope(void) {
    int local = 10;
    printf("local=%d, global=%d\n", local, global);
}
show_scope();

## 6. 递归函数

递归需要**终止条件**和**向终止条件推进的步骤**。先明确允许的输入，再讨论算法。

下面用 `long long` 返回阶乘，仅接受 0..20；20! 在 C17 保证的 long long 范围内，21! 不在本例允许范围。-1 明确表示无效输入，不能当成正常结果。斐波那契示例限制 1..20，避免指数级重复计算拖慢浏览器。

In [ ]:
long long factorial(int n) {
    if (n < 0 || n > 20) { return -1; }
    if (n == 0) { return 1; }
    return n * factorial(n - 1);
}

int fib(int n) {
    if (n < 1 || n > 20) { return -1; }
    if (n <= 2) { return 1; }
    return fib(n - 1) + fib(n - 2);
}

{
    printf("0!=%lld 5!=%lld\n", factorial(0), factorial(5));
    printf("20!=%lld\n", factorial(20));
    printf("无效输入: %lld %lld\n", factorial(-1), factorial(21));
    printf("fib(10)=%d\n", fib(10));
}

## 实验 1：声明、定义与调用的关系

声明告诉编译器参数和返回值类型，定义提供函数体。在同一个单元中先声明、再定义，然后调用；本地程序可以把定义放在 main 后面，只要调用处已能看到声明。

**预测**调用结果；修改声明而不修改定义，会发生什么？先在纸上判断，不要把不匹配版本混入后续运行。

In [ ]:
int larger(int left, int right);
int larger(int left, int right) { return left > right ? left : right; }
printf("较大值: %d\n", larger(85, 90));

## 实验 2：数组参数必须带长度

形参 `const int scores[]` 会调整为指针参数，函数不会自动获得调用者的数组长度。不要在函数内用 `sizeof(scores) / sizeof(scores[0])` 求人数。

`const` 表示此函数不通过该指针修改元素。调用者仍要保证实际数组至少有 count 个元素；函数无法从地址推断容量。

下面是贯穿案例的统计函数：最多 100 人，成绩必须在 0..100。`valid == 0` 明确表示输入无效，调用者必须检查。

In [ ]:
struct GradeSummary {
    int valid;
    double average;
    size_t passed;
};

// 不修改输入；count 必须对应调用者提供的实际元素数。
struct GradeSummary summarize(const int scores[], size_t count) {
    struct GradeSummary result = {0, 0.0, 0};
    if (scores == NULL || count == 0 || count > 100) { return result; }
    int total = 0;
    size_t passed = 0;
    for (size_t i = 0; i < count; ++i) {
        if (scores[i] < 0 || scores[i] > 100) { return result; }
        total += scores[i];
        if (scores[i] >= 60) { ++passed; }
    }
    result.valid = 1;
    result.average = (double)total / count;
    result.passed = passed;
    return result;
}

## 贯穿案例 6：分离统计与显示

先预测平均分与及格人数，再运行。统计函数只返回结果，显示由调用代码完成。将第三人成绩改成 -1，应该进入错误分支，不能显示部分统计结果。

拆分依据：人数与成绩校验、累计求和属于统计函数；显示文字属于调用代码。用参数传入数据，用返回值报告结果，避免依赖全局成绩数组。

In [ ]:
{
    const int scores[] = {85, 90, 58};
    size_t count = sizeof scores / sizeof scores[0];
    struct GradeSummary result = summarize(scores, count);
    if (!result.valid) { printf("无法统计：人数或成绩无效\n"); }
    else { printf("平均分: %.2f, 及格: %zu/%zu\n", result.average, result.passed, count); }
}

## 边界实验：空数据与非法成绩

以下是有意传入的无效输入，函数应明确拒绝，不访问空指针，也不执行除以零。预测两个 valid 字段的值后运行。

In [ ]:
{
    const int invalid[] = {85, -1};
    struct GradeSummary empty = summarize(NULL, 0);
    struct GradeSummary bad = summarize(invalid, 2);
    printf("空数据有效=%d 非法成绩有效=%d\n", empty.valid, bad.valid);
}

## 实验 3：展开递归调用

先写下 `factorial(3)` 的调用链：3 → 2 → 1 → 0，然后从 0! = 1 向上返回。请用循环实现相同的 0..20 阶乘接口，比较输入 0、5、20、-1、21。

递归不一定更快；朴素 fib 会重复计算相同子问题。对连续生成斐波那契数列，可保留前两个结果并迭代。

## 分层练习

### 1. 读程序
函数接收 `int x`，在内部把 x 改成10，为什么调用者的 a 不变？

<details><summary>提示：先自己尝试</summary>

区分对象和对象值的副本。

</details>

<details><summary>参考思路与自查</summary>

形参是副本。即使传指针，也仍复制指针值，只是能通过该地址访问同一个对象。

</details>

### 2. 改错
函数 `int length(int a[]) { return sizeof(a) / sizeof(a[0]); }` 为什么不能求数组长度？

<details><summary>提示：先自己尝试</summary>

数组参数在函数中是什么类型？

</details>

<details><summary>参考思路与自查</summary>

它是指针参数；由调用者计算实际数组长度并显式传入。

</details>

### 3. 编程
写 `int is_passing(int score)`，0..59 返回0，60..100 返回1，非法成绩返回-1。

<details><summary>提示：先自己尝试</summary>

先检查有效范围，再进行等级判断。

</details>

<details><summary>参考思路与自查</summary>

至少测试 -1、0、59、60、100、101，预期 -1、0、0、1、1、-1。

</details>

### 4. 综合扩展
为 GradeSummary 增加最高分字段，保持输入数组不变，并处理空数据。

<details><summary>提示：先自己尝试</summary>

只有确认至少一个有效元素后才能使用第0项初始化最高分。

</details>

<details><summary>参考思路与自查</summary>

样例最高分90；单人成绩60的最高分为60；空数据仍 valid=0，不伪造最高分。

</details>

## 综合练习：完成成绩统计器

给定最多100人成绩，返回合法性、平均分、最高分与及格人数。默认数据始终是 Alice=85、Bob=90、Cathy=58；统计函数只需要成绩数组，姓名属于展示层。

**接口约定**：

- `count` 必须与调用者实际提供的元素数一致；函数无法从指针推断容量。
- 空指针、人数为0或超过100、任意成绩不在0..100，均返回 valid=0，其余字段为0。
- 全部合法时 valid=1；不修改输入，不在统计函数内输出。
- 非法成绩使整个报告无效。本题与第4章“过滤无效成绩”练习采用不同规则，不能混用。

**完成顺序**：先校验 → 再累计 → 返回结果 → 由调用者显示。不要让部分累计结果伪装成完整报告。

下面是待完成的函数，不是参考答案。初次运行验收单元会出现 FAIL；完成后应全部 PASS。修改函数定义后重启内核并从头运行。

<details><summary>一级提示：组织数据</summary>

用一个结构体返回多个结果。先准备全零的无效结果；只有所有成绩都合法才填入最终结果。

</details>

<details><summary>二级提示：避免边界错误</summary>

访问数组前拒绝空指针与无效人数；累计整数总分，最后进行浮点除法；因为有效成绩非负，最高分可从0开始。不在发现非法成绩前逐步更新将要返回的有效结果。

</details>

In [ ]:
struct Report {
    int valid;
    double average;
    int highest;
    size_t passed;
};

struct Report report_grade(const int scores[], size_t count) {
    // TODO: 实现接口约定；删除下面仅用于占位的语句。
    (void)scores;
    (void)count;
    struct Report result = {0, 0.0, 0, 0};
    return result;
}

### 运行验收：定位失败用例

下面检查默认三人成绩、单人成绩、空数据、负分、超上限、空指针与超人数。每行显示实际报告与期望报告；只检查到小数点后若干位所需的绝对误差，容差适用于本题0..100的平均分。

还应自己补充 `{0,100}`（平均50、最高100、及格1）和 `{59,60}`（平均59.5、最高60、及格1）。自动验收通过后，口头解释一次“为何不能用 scores[0] 无条件初始化最高分”。

In [ ]:
{
    const int normal[] = {85, 90, 58};
    const int single[] = {60};
    const int negative[] = {85, -1};
    const int too_high[] = {101};
    struct ReportCase {
        const char *label;
        const int *scores;
        size_t count;
        struct Report expected;
    };
    const struct ReportCase cases[] = {
        {"三人成绩", normal, 3, {1, 233.0 / 3, 90, 2}},
        {"单人成绩", single, 1, {1, 60.0, 60, 1}},
        {"空数据", NULL, 0, {0, 0.0, 0, 0}},
        {"负分", negative, 2, {0, 0.0, 0, 0}},
        {"超上限", too_high, 1, {0, 0.0, 0, 0}},
        {"空指针", NULL, 1, {0, 0.0, 0, 0}},
        {"超人数", normal, 101, {0, 0.0, 0, 0}}
    };
    const size_t case_count = sizeof cases / sizeof cases[0];
    size_t passed = 0;
    for (size_t i = 0; i < case_count; ++i) {
        struct Report actual = report_grade(cases[i].scores, cases[i].count);
        struct Report expected = cases[i].expected;
        double difference = actual.average - expected.average;
        int ok = actual.valid == expected.valid && actual.highest == expected.highest
            && actual.passed == expected.passed && difference > -0.000001 && difference < 0.000001;
        if (ok) { ++passed; }
        printf("%s %s: 实际=(%d,%.2f,%d,%zu) 期望=(%d,%.2f,%d,%zu)\n",
               ok ? "PASS" : "FAIL", cases[i].label,
               actual.valid, actual.average, actual.highest, actual.passed,
               expected.valid, expected.average, expected.highest, expected.passed);
    }
    printf("通过 %zu/%zu；全部通过后再增加自己的边界样例。\n", passed, case_count);
}

## 第二阶段：从空白文件完成程序

完成统计函数后，收起模板和参考答案，新建空白 report.c。只依据下面的要求独立实现；允许查标准库函数用法，不复制本章完整程序。

**输入与职责**：第一项为人数1..100，随后读取对应数量的整数成绩0..100。假定数字文本可表示为 int；读取失败或成绩非法时报告错误并以非零状态退出。用动态数组保存成绩，用独立统计函数计算平均分、最高分与及格人数，由 main 负责输入输出与释放。

| 输入 | 验收要求 |
| --- | --- |
| `3 85 90 58` | 平均77.67，最高90，及格2/3 |
| `1 60` | 平均60.00，最高60，及格1/1 |
| `2 0 100` | 平均50.00，最高100，及格1/2 |
| `2 59 60` | 平均59.50，最高60，及格1/2 |
| `0` 或 `101` | 拒绝人数，不访问成绩数组 |
| `2 85 -1` | 拒绝非法成绩，不显示部分统计报告 |
| `2 85 abc` 或只输入 `2 85` 后结束输入 | 明确报告读取失败 |

排版可自行设计，但信息与数值必须正确。用 `cc -std=c17 -Wall -Wextra -Wpedantic -Werror report.c -o report` 编译；修复所有诊断后再运行这些输入。

**完成标准**：从空白文件完成编译与运行；正常和错误用例均符合约定；能指出释放内存的每条路径，并解释为什么只通过默认三人成绩还不够。提交源文件与一份简短的实际运行记录。

迁移挑战：保持输入输出分离的组织方式，把一份自己的统计程序改成每日温度统计。先自行明确允许范围与统计指标，再设计用例，检查自己掌握的是方法而非某一组成绩答案。

## 本地实践：综合练习参考程序

先完成在线作答并通过验收，再展开。保存为 `chapter06.c`，用 `cc -std=c17 -Wall -Wextra -Wpedantic -Werror chapter06.c -o chapter06` 编译，运行 `./chapter06`（Windows 使用 chapter06.exe）。输入第一行为人数，例如3，随后是 `85 90 58`。

本练习假定数字文本可表示为 int，检查转换失败和业务范围；不处理任意长整数文本，也不要求拒绝额外尾随文本。

注意本地程序的顺序：结构体和函数声明 → main 中调用 → 函数定义。在线核对答案时，只将 report_grade 的定义替换到作答单元，保留验收单元，再重启运行；不要把完整 main 放进 Notebook。

<details><summary>完整参考答案：包含输入、动态内存、统计与输出</summary>

```c
#include <stdio.h>
#include <stdlib.h>

struct Report {
    int valid;
    double average;
    int highest;
    size_t passed;
};

struct Report report_grade(const int scores[], size_t count);

int main(void) {
    int count;
    if (scanf("%d", &count) != 1 || count < 1 || count > 100) {
        printf("人数无效\n");
        return 1;
    }
    int *scores = malloc((size_t)count * sizeof *scores);
    if (scores == NULL) { printf("分配失败\n"); return 1; }
    for (int i = 0; i < count; ++i) {
        if (scanf("%d", &scores[i]) != 1) {
            printf("成绩读取失败\n");
            free(scores);
            return 1;
        }
    }
    struct Report result = report_grade(scores, (size_t)count);
    free(scores);
    if (!result.valid) { printf("成绩无效\n"); return 1; }
    printf("平均分=%.2f 最高分=%d 及格=%zu/%d\n", result.average, result.highest, result.passed, count);
    return 0;
}

struct Report report_grade(const int scores[], size_t count) {
    struct Report result = {0, 0.0, 0, 0};
    if (scores == NULL || count == 0 || count > 100) { return result; }
    int total = 0;
    int highest = 0;
    size_t passed = 0;
    for (size_t i = 0; i < count; ++i) {
        if (scores[i] < 0 || scores[i] > 100) { return result; }
        total += scores[i];
        if (scores[i] > highest) { highest = scores[i]; }
        if (scores[i] >= 60) { ++passed; }
    }
    result.valid = 1;
    result.average = (double)total / count;
    result.highest = highest;
    result.passed = passed;
    return result;
}
```

</details>

## 小结

清晰的函数接口说明输入、返回值和无效情况。数组参数需配合长度；递归需终止条件与有效范围。优先让计算函数只读取输入并返回结果，让调用者负责输入输出。